# Stage 11 — Option buying ranking

Full-universe backtest (notebook 17, WITH the 4H hard filter applied) showed 70.6% of qualified instruments with a positive avg R, mean 0.198, median 0.170. This ranks instruments the same way the other four strategies are ranked (`strategy_scoring.py`), so the shortlist is the instruments where this specific signal+filter+lifecycle actually worked, not the whole universe.

**`min_trades=20` here, not the 30 used for debit spread** - the 4H filter roughly halves signal frequency, so per-instrument closed_trades over the same window now tops out around 27 with a mean of ~19; a 30-trade bar would leave nothing ranked. See `score_for_option_buying()`'s docstring for the full reasoning.

median_r_achieved is weighted highest (not avg_r_achieved) - same median-over-mean preference as the rest of this project's stats.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.strategy_scoring import score_for_option_buying, save_ranking

comparison = pd.read_excel("../exports/OPTION_BUYING_COMPARISON.xlsx", sheet_name="Comparison")
comparison.shape

(224, 8)

In [2]:
ranked, watchlist = score_for_option_buying(comparison)
ranked.head(20)

,symbol,option_buying_score,outlier_reliance_flag,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held,pct_trades_trailing_activated
0,SWIGGY,96.8,,20,80.0,1.10,0.94,6.0,75.0
1,KPITTECH,92.4,,38,76.3,0.60,0.66,6.5,71.1
2,RVNL,91.8,,43,74.4,0.83,0.74,7.0,72.1
3,ADANIENSOL,91.6,,36,63.9,0.45,0.44,5.0,61.1
4,GMRAIRPORT,91.6,,34,73.5,0.52,0.39,6.0,73.5
5,GODREJPROP,91.5,,33,72.7,0.49,0.41,6.0,69.7
6,PFC,91.3,,28,71.4,0.88,0.72,7.0,71.4
7,SUZLON,91.1,,40,62.5,0.75,0.53,6.0,60.0
8,MAHABANK,89.3,,35,68.6,0.36,0.42,6.0,62.9
9,COFORGE,89.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.35R),43,65.1,0.67,0.32,6.0,65.1


In [3]:
# How many top-20 candidates carry the outlier-reliance warning?
print(f"Instruments passing min_trades filter: {len(ranked)} of {len(comparison)}")
print(f"Flagged in top 20: {(ranked.head(20)['outlier_reliance_flag'] != '').sum()}")
ranked[ranked["outlier_reliance_flag"] != ""].head(10)

Instruments passing min_trades filter: 214 of 224
Flagged in top 20: 1


,symbol,option_buying_score,outlier_reliance_flag,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held,pct_trades_trailing_activated
9,COFORGE,89.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.35R),43,65.1,0.67,0.32,6.0,65.1
23,RECLTD,83.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.36R),32,65.6,0.58,0.22,6.0,62.5
28,POWERINDIA,81.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.36R),45,64.4,0.61,0.25,7.0,57.8
29,SHRIRAMFIN,79.9,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.61R),38,68.4,0.81,0.20,7.0,63.2
31,VOLTAS,78.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.31R),31,61.3,0.59,0.28,8.0,61.3
32,PAYTM,76.9,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.41R),35,62.9,0.61,0.20,7.0,57.1
38,INFY,74.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.45R),42,61.9,0.65,0.20,8.0,57.1
39,OIL,74.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.42R),34,61.8,0.60,0.18,7.0,61.8
40,BANKINDIA,74.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.30R),34,61.8,0.52,0.22,7.5,61.8
41,DIVISLAB,73.7,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.34R),31,61.3,0.53,0.19,7.0,58.1


In [4]:
save_ranking(ranked, "RANKING_option_buying.xlsx")

Saved ranking to /Users/hgmac/Documents/3_trading/Quant/exports/RANKING_option_buying.xlsx


'/Users/hgmac/Documents/3_trading/Quant/exports/RANKING_option_buying.xlsx'

In [5]:
# Instruments below min_trades - not blended into the percentile ranking,
# but not dropped either. Sorted by avg_r_achieved. Re-check at the next
# periodic refresh - trade count grows on its own as more calendar time passes.
print(f"Instruments on watchlist (below min_trades): {len(watchlist)}")
watchlist

Instruments on watchlist (below min_trades): 10


,symbol,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held,pct_trades_trailing_activated
0,ATHERENERG,18,66.7,0.24,0.16,5.0,66.7
1,WAAREEENER,19,63.2,0.23,0.19,6.0,57.9
2,ENRIN,14,50.0,0.20,-0.40,7.5,42.9
3,HYUNDAI,17,52.9,-0.17,0.01,7.0,52.9
4,COROMANDEL,0,NaN,NaN,NaN,NaN,NaN
5,CUB,0,NaN,NaN,NaN,NaN,NaN
6,DATAPATTNS,0,NaN,NaN,NaN,NaN,NaN
7,KARURVYSYA,0,NaN,NaN,NaN,NaN,NaN
8,NAVINFLUOR,0,NaN,NaN,NaN,NaN,NaN
9,SYRMA,0,NaN,NaN,NaN,NaN,NaN


In [6]:
save_ranking(watchlist, "WATCHLIST_option_buying.xlsx")

Saved ranking to /Users/hgmac/Documents/3_trading/Quant/exports/WATCHLIST_option_buying.xlsx


'/Users/hgmac/Documents/3_trading/Quant/exports/WATCHLIST_option_buying.xlsx'